# Flickr30k Hybrid paper tables

Xuất bảng ×100 và nội dung kết quả âm cho báo cáo khoa học.

## Kaggle input bắt buộc

1. `F30K_T1S1_Hybrid_Aggregate`
2. `F30K_T1S1_Hybrid_Closeout`
3. `F30K_T1S1_SeqKD_Aggregate`
4. `F30K_T1S1_DenseMap_Aggregate`
5. `F30K_Final_Audit`

Runtime: **CPU / Accelerator None / Internet Off**. Notebook không đọc ảnh,
không đọc Held-out Test, không huấn luyện và không xuất ONNX/INT8.

## Kiểm toán và xuất artifact

In [1]:
import csv
import hashlib
import json
import os
from datetime import datetime, timezone
from pathlib import Path

EXPERIMENT_ID = "F30K_Hybrid_Paper_Tables"
INPUT_ROOT = Path(os.environ.get("KLCN_INPUT_ROOT", "/kaggle/input"))
WORK_ROOT = Path(os.environ.get("KLCN_OUTPUT_ROOT", "/kaggle/working")) / EXPERIMENT_ID
WORK_ROOT.mkdir(parents=True, exist_ok=True)

def sha256(path, chunk=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while block := handle.read(chunk):
            digest.update(block)
    return digest.hexdigest()

def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def unique_named(name):
    matches = sorted(p for p in INPUT_ROOT.rglob(name) if p.is_file())
    assert len(matches) == 1, (f"Cần đúng một {name}", [str(p) for p in matches])
    return matches[0]

def write_json(path, payload):
    temporary = Path(path).with_suffix(Path(path).suffix + ".tmp")
    temporary.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    os.replace(temporary, path)

paths = {
    "hybrid": unique_named("F30K_T1S1_Hybrid_Aggregate_RESULT.json"),
    "closeout": unique_named("F30K_T1S1_Hybrid_Closeout_RESULT.json"),
    "sequence": unique_named("F30K_T1S1_SeqKD_Aggregate_RESULT.json"),
    "dense": unique_named("F30K_T1S1_DenseMap_Aggregate_RESULT.json"),
    "final_audit": unique_named("F30K_Final_Audit_RESULT.json"),
}
objects = {name: read_json(path) for name, path in paths.items()}
hybrid = objects["hybrid"]
closeout = objects["closeout"]
sequence = objects["sequence"]
dense = objects["dense"]
final_audit = objects["final_audit"]

for obj in (hybrid, sequence, dense):
    assert obj["status"] == "complete"
    assert obj["run_count"] == 3
    assert obj["seeds"] == [42, 123, 2026]
    assert obj["decision"] == "STOP_VALIDATION_NO_GAIN"
assert closeout["conclusion"] == "STOP_VALIDATION_NO_GAIN"
assert closeout["onnx_or_int8_authorized_for_hybrid"] is False
assert final_audit["scientific_evaluation_complete"] is True

metrics = ("BLEU_1", "BLEU_4", "METEOR", "ROUGE_L", "CIDEr_D")
rows = []
baseline_summary = hybrid["summary"]
baseline_row = {
    "method": "S1 baseline",
    "role": "baseline",
    "CE_mean_raw": round(baseline_summary["ce_loss"]["baseline_mean"], 6),
    "CE_sample_sd_raw": round(baseline_summary["ce_loss"]["baseline_sample_sd"], 6),
    "decision": "REFERENCE",
}
for metric in metrics:
    baseline_row[f"{metric}_mean_x100"] = round(
        baseline_summary[metric]["baseline_mean"] * 100, 2
    )
    baseline_row[f"{metric}_sample_sd_x100"] = round(
        baseline_summary[metric]["baseline_sample_sd"] * 100, 2
    )
baseline_row["delta_CIDEr_D_x100"] = 0.0
rows.append(baseline_row)

for label, obj in (
    ("Sequence KD", sequence),
    ("Dense Mapped Logit-KD", dense),
    ("Hybrid Sequence + DenseMap", hybrid),
):
    summary = obj["summary"]
    row = {
        "method": label,
        "role": "KD",
        "CE_mean_raw": round(summary["ce_loss"]["kd_mean"], 6),
        "CE_sample_sd_raw": round(summary["ce_loss"]["kd_sample_sd"], 6),
        "decision": obj["decision"],
    }
    for metric in metrics:
        row[f"{metric}_mean_x100"] = round(summary[metric]["kd_mean"] * 100, 2)
        row[f"{metric}_sample_sd_x100"] = round(
            summary[metric]["kd_sample_sd"] * 100, 2
        )
    row["delta_CIDEr_D_x100"] = round(
        summary["CIDEr_D"]["delta_mean"] * 100, 2
    )
    rows.append(row)

method_table = WORK_ROOT / "table_hybrid_methods_x100.csv"
with method_table.open("w", encoding="utf-8-sig", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
    writer.writeheader()
    writer.writerows(rows)

seed_rows = []
for run in hybrid["runs"]:
    seed_rows.append({
        "seed": run["seed"],
        "baseline_best_epoch": run["baseline_best_epoch"],
        "hybrid_best_epoch": run["kd_best_epoch"],
        "baseline_CIDEr_D_x100": round(run["baseline_CIDEr_D"] * 100, 2),
        "hybrid_CIDEr_D_x100": round(run["kd_CIDEr_D"] * 100, 2),
        "delta_CIDEr_D_x100": round(run["delta_CIDEr_D"] * 100, 2),
        "hybrid_BLEU_1_x100": round(run["kd_BLEU_1"] * 100, 2),
        "hybrid_BLEU_4_x100": round(run["kd_BLEU_4"] * 100, 2),
        "hybrid_METEOR_x100": round(run["kd_METEOR"] * 100, 2),
        "hybrid_ROUGE_L_x100": round(run["kd_ROUGE_L"] * 100, 2),
    })
seed_table = WORK_ROOT / "table_hybrid_seeds_x100.csv"
with seed_table.open("w", encoding="utf-8-sig", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=list(seed_rows[0]))
    writer.writeheader()
    writer.writerows(seed_rows)

ci = hybrid["combined_paired_bootstrap"]
result_payload = {
    "schema_version": "f30k_hybrid_paper_tables_v1",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "status": "complete_negative_result_documented",
    "experiment_id": EXPERIMENT_ID,
    "metric_display_scale": "x100_except_CE",
    "methods_documented": [row["method"] for row in rows],
    "hybrid_mean_CIDEr_D_x100": round(hybrid["summary"]["CIDEr_D"]["kd_mean"] * 100, 2),
    "hybrid_delta_CIDEr_D_x100": round(hybrid["summary"]["CIDEr_D"]["delta_mean"] * 100, 2),
    "hybrid_delta_ci95_x100": [
        round(ci["ci95_lower"] * 100, 2),
        round(ci["ci95_upper"] * 100, 2),
    ],
    "conclusion": "Hybrid did not improve S1 on Flickr30k; branch closed.",
    "existing_locked_model": closeout["existing_locked_model"],
    "model_registry_change_authorized": False,
    "test_set_used": False,
    "next_phase_authorized": "F30K_Final_Snapshot_V2",
    "source_sha256": {name: sha256(path) for name, path in paths.items()},
}
result_path = WORK_ROOT / f"{EXPERIMENT_ID}_RESULT.json"
write_json(result_path, result_payload)

report_path = WORK_ROOT / "REPORT.md"
report_path.write_text(
    "# Hybrid Sequence–DenseMap KD trên Flickr30k\n\n"
    "Hybrid KD đạt CIDEr-D trung bình "
    f"**{result_payload['hybrid_mean_CIDEr_D_x100']:.2f}**, thấp hơn baseline "
    f"**{abs(result_payload['hybrid_delta_CIDEr_D_x100']):.2f} điểm**. "
    "Khoảng tin cậy bootstrap ghép cặp 95% của hiệu số là "
    f"**[{result_payload['hybrid_delta_ci95_x100'][0]:+.2f}; "
    f"{result_payload['hybrid_delta_ci95_x100'][1]:+.2f}]**. "
    "Cả ba seed đều giảm, vì vậy nhánh được đóng và không chuyển sang "
    "ONNX/INT8. Kết quả âm này cần được báo cáo đầy đủ, không loại khỏi bảng.\n\n"
    "Mô hình triển khai đã khóa vẫn là **T3→S3 Direct Logit-KD INT8**.\n",
    encoding="utf-8",
)

manifest_items = []
for artifact in (result_path, method_table, seed_table, report_path):
    manifest_items.append({
        "relative_path": artifact.name,
        "size_bytes": artifact.stat().st_size,
        "sha256": sha256(artifact),
    })
write_json(WORK_ROOT / "MANIFEST.json", {
    "schema_version": "f30k_hybrid_paper_manifest_v1",
    "experiment_id": EXPERIMENT_ID,
    "status": "complete",
    "artifacts": manifest_items,
    "test_set_used": False,
})

print("Status                  :", result_payload["status"])
print("Hybrid CIDEr-D x100     :", result_payload["hybrid_mean_CIDEr_D_x100"])
print("Delta x100              :", result_payload["hybrid_delta_CIDEr_D_x100"])
print("95% CI x100             :", result_payload["hybrid_delta_ci95_x100"])
print("Registry change         : NOT AUTHORIZED")

Status                  : complete_negative_result_documented
Hybrid CIDEr-D x100     : 38.31
Delta x100              : -9.24
95% CI x100             : [-11.53, -6.94]
Registry change         : NOT AUTHORIZED
